# ADRETM · Clasificación con LLM (etapa 1)

Clasifica las publicaciones y tesis de la **base unificada** usando un catálogo cerrado.
- **El módulo** `clasificador_llm.py` se descarga de GitHub (`DatosLabUPLA/datos-analitca-educativa-web`, carpeta `custom_files/adretm/clasificador`) en la celda 1. Para usar una versión nueva, basta con volver a ejecutar esa celda; GitHub puede tardar unos minutos en servir un archivo recién subido.
- **El catálogo** vive en la hoja, en las pestañas `catalogo_dimensiones` y `catalogo_clasificacion`, y se edita ahí. Los CSV del repositorio solo sirven como semilla inicial.

## Qué se clasifica en cada corrida
- **Registros nuevos**: los que no tienen fila en `control_clasificacion` para esa pasada.
- **Registros marcados**: los que tienen **`reclasificar = Sí`** en `control_clasificacion`.
- **Registros cuyo texto cambió**: solo si `SI_CAMBIA_TEXTO = True`. Si no, la celda solo avisa cuántos son.
- **Lo que esté en `IDS`**, si `FORZAR = True`.

Al reclasificar un registro, sus etiquetas LLM *Pendientes* anteriores pasan a *Reemplazada*. Las *Aceptadas* y *Rechazadas* no se tocan, porque son decisiones humanas. Los registros que dieron error se vuelven a intentar en la siguiente corrida.

## Pestañas que escribe
| Pestaña | Contenido |
|---|---|
| `etiquetas` | Una fila por etiqueta. En `estado` eliges Pendiente, Aceptada, Rechazada o Reemplazada |
| `control_clasificacion` | Una fila por registro y pasada: cuándo se clasificó, una huella del texto, `reclasificar` y `nota` |
| `propuestas_catalogo` | Claves nuevas que sugiere el LLM, para que las revises |

## Celdas
| Celda | Qué hace | ¿Cuándo? |
|---|---|---|
| 1 | Autenticación, descarga del módulo, cliente OpenAI | Siempre |
| 2 | Sube el catálogo semilla a la hoja | **Solo la primera vez** |
| 3 | Lee y valida el catálogo | Siempre, y después de editarlo |
| 4 | Configuración y vista previa del prompt (sin costo) | Siempre |
| 5 | Corrida: elige qué se clasifica | En cada corrida |
| 6 | Guarda el resultado en la hoja | Después de revisar la celda 5 |

# 1 Autenticación y módulo

In [ ]:
# Autenticación, módulo desde GitHub y cliente OpenAI
# Secretos de Colab (ícono de llave): OPENAI_API_KEY, y GITHUB_TOKEN solo si el repositorio es privado.

from google.colab import auth, userdata
auth.authenticate_user()

import os, sys, importlib, requests
import gspread
import pandas as pd
from io import StringIO
from google.auth import default
from IPython.display import display
from openai import OpenAI

REPO = 'DatosLabUPLA/datos-analitca-educativa-web'
RAMA = 'main'
CARPETA_REPO = 'custom_files/adretm/clasificador'
DIR_LOCAL = '/content/clasificador'

try:
    token = userdata.get('GITHUB_TOKEN')
except Exception:
    token = ''
CABECERAS = {'Authorization': f'token {token}'} if token else {}

def desde_github(ruta):
    """Texto de un archivo del repositorio (ruta relativa a CARPETA_REPO)."""
    url = f'https://raw.githubusercontent.com/{REPO}/{RAMA}/{CARPETA_REPO}/{ruta}'
    r = requests.get(url, headers=CABECERAS, timeout=30)
    r.raise_for_status()
    return r.text

os.makedirs(DIR_LOCAL, exist_ok=True)
with open(f'{DIR_LOCAL}/clasificador_llm.py', 'w', encoding='utf-8') as f:
    f.write(desde_github('clasificador_llm.py'))
if DIR_LOCAL not in sys.path:
    sys.path.insert(0, DIR_LOCAL)
import clasificador_llm as cl
importlib.reload(cl)

try:   # último cambio del módulo, para saber qué versión se está usando
    c = requests.get(f'https://api.github.com/repos/{REPO}/commits',
                     params={'path': f'{CARPETA_REPO}/clasificador_llm.py', 'sha': RAMA, 'per_page': 1},
                     headers=CABECERAS, timeout=30).json()[0]
    commit = f"{c['sha'][:7]} {c['commit']['author']['date'][:10]} {c['commit']['message'].splitlines()[0]}"
except Exception:
    commit = '(no disponible)'

# Caché de respuestas en Drive (opcional): evita pagar dos veces la misma consulta
# si una corrida se interrumpe antes de guardar. None para no usarlo.
RUTA_CACHE = '/content/drive/MyDrive/ADRETM/cache_llm.jsonl'
if RUTA_CACHE:
    from google.colab import drive
    drive.mount('/content/drive')
    os.makedirs(os.path.dirname(RUTA_CACHE), exist_ok=True)

creds, _ = default()
gc = gspread.authorize(creds)
CLAVE_BASE = '1XgvA7ZCIizoZARLFf7zAHviizU1SRymDtVKRWUKoZzI'
sh_base = gc.open_by_key(CLAVE_BASE)
client = OpenAI(api_key=userdata.get('OPENAI_API_KEY'))
print(f'Módulo {cl.__version__} · último cambio: {commit} · base: {sh_base.title}')

# 2 Subir el catálogo semilla (solo la primera vez)

In [ ]:
# Sube los CSV semilla del repositorio a las pestañas del catálogo. Si las pestañas ya existen, NO las toca
# (sobrescribir=True las reemplaza y se pierde lo editado en la hoja).

semilla = cl.Catalogo(pd.read_csv(StringIO(desde_github('catalogos/etm/catalogo_dimensiones.csv')), dtype=str, keep_default_na=False),
                      pd.read_csv(StringIO(desde_github('catalogos/etm/catalogo_clasificacion.csv')), dtype=str, keep_default_na=False))
cl.escribir_tabla(sh_base, 'catalogo_dimensiones', semilla.tabla_dimensiones[cl.COLS_DIMENSIONES])
cl.escribir_tabla(sh_base, 'catalogo_clasificacion', semilla.tabla_claves[cl.COLS_CATALOGO])

# 3 Leer y validar el catálogo

In [ ]:
# Lee el catálogo desde la hoja. Si hay errores, la celda se detiene y los lista.

cat = cl.Catalogo.desde_hoja(sh_base)
display(cat.resumen())
print(f'{len(cat.avisos)} avisos')
for a in cat.avisos[:30]:
    print('  ·', a)

# 4 Configuración y vista previa (sin costo)

In [ ]:
# Qué ve el LLM de cada entidad: {'Etiqueta en el prompt': 'columna de la pestaña'}
# Verificar que los nombres de columna coincidan EXACTAMENTE con los encabezados de la hoja.

CONTEXTO = ('Los trabajos pertenecen a la red ADRETM, que investiga en Didáctica de la Matemática con la teoría de '
            'los Espacios de Trabajo Matemático (ETM; en inglés Mathematical Working Spaces, MWS; en francés '
            'Espace de Travail Mathématique). Pueden estar en español, francés, inglés o portugués.')

CAMPOS = {
    'publicaciones': {'Título': 'titulo', 'Títulos alternativos': 'titulos_alternativos', 'Tipo': 'tipo',
                      'Año': 'anio', 'Fuente': 'fuente', 'Palabras clave': 'palabras_clave', 'Resumen': 'resumen'},
    'tesis': {'Título': 'titulo', 'Tipo': 'tipo', 'Año de defensa': 'anio_defensa', 'Universidad': 'universidad'},
}
COL_ID = {'publicaciones': 'id_publicacion', 'tesis': 'id_tesis'}

clf = cl.Clasificador(cat, client, contexto=CONTEXTO, modelo='gpt-4o-mini', ruta_cache=RUTA_CACHE)
registros = {ent: cl.leer_tabla(sh_base, ent) for ent in CAMPOS}
previas = cl.leer_tabla(sh_base, 'etiquetas')
control = cl.leer_tabla(sh_base, 'control_clasificacion')

for ent, df_ent in registros.items():
    faltan = [c for c in list(CAMPOS[ent].values()) + [COL_ID[ent]] if c not in df_ent.columns]
    print(f'{ent}: {len(df_ent)} filas' + (f' · ⚠️ columnas que no existen: {faltan}' if faltan else ''))
print('etiquetas previas:', len(previas), '· filas de control:', len(control))

# vista previa de la pasada 2 (se simula que el registro usa el ETM)
ejemplo = registros['publicaciones'].query("resumen != ''").iloc[0]
_ = clf.vista_previa(ejemplo, CAMPOS['publicaciones'], pasada=2,
                     etiquetas_registro={'ETM_RELACION': {'usa_etm'}})

# 5 Corrida

In [ ]:
# Clasifica lo que corresponde y muestra el resultado. NO escribe en la hoja (eso lo hace la celda 6).
# Ejemplos de IDS:
#   tesis sin clasificar: registros['tesis'].query("etm_estado == 'Sin clasificar'")['id_tesis'].tolist()
#   piloto: una lista de ~20 IDs escrita a mano

ENTIDAD = 'publicaciones'      # 'publicaciones' o 'tesis'
PASADA = 1                     # 1 = relación con el ETM e idioma · 2 = el resto (solo para usa_etm)
IDS = None                     # None = todos
FORZAR = False                 # True = reclasifica todo lo de IDS aunque ya esté hecho
SI_CAMBIA_TEXTO = False        # True = reclasifica también los registros cuyo texto cambió
LIMITE = 3                     # para probar; None = sin límite

res = clf.clasificar(registros[ENTIDAD], COL_ID[ENTIDAD], CAMPOS[ENTIDAD], PASADA, entidad=ENTIDAD,
                     previas=previas, control=control, ids=IDS, forzar=FORZAR,
                     si_cambia_texto=SI_CAMBIA_TEXTO, limite=LIMITE)
print(res)
display(res.etiquetas[['id_registro', 'dimension', 'clave', 'rol', 'confianza', 'evidencia', 'observacion']])
if len(res.propuestas):
    display(res.propuestas[['id_registro', 'dimension', 'clave_propuesta', 'etiqueta_es', 'justificacion']])
if len(res.errores):
    display(res.errores)

# 6 Guardar en la base

In [ ]:
# Guarda la corrida de la celda 5. Si ya se guardó, no la duplica.

cl.guardar_corrida(sh_base, res)
previas = cl.leer_tabla(sh_base, 'etiquetas')
control = cl.leer_tabla(sh_base, 'control_clasificacion')
print('etiquetas:', len(previas), '· filas de control:', len(control))